# 🏷️ Project 8.1 — Sensor ID Registry (Hash Map from Scratch)
**DSA for Mechatronics · Week 08 — Hash Tables & Dictionaries**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
from collections import Counter, defaultdict, OrderedDict
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory sensor network keeps a registry that maps **sensor IDs** (strings) to
metadata records (zone, type, calibration date).

You will build a hash map **from scratch** using **separate chaining**:
each slot holds a linked list of (key, value) pairs that hashed to the same index.
Watch how the **load factor** (n / capacity) grows and triggers a **resize**.


## Step 1 — Build the hash map with separate chaining

In [ ]:
# ── Personalised parameters ───────────────────────────────────────
N_SENSORS     = random.randint(24, 40)
INIT_CAPACITY = random.choice([8, 16, 32])
LOAD_THRESH   = round(random.uniform(0.65, 0.80), 2)
N_DELETE      = random.randint(3, 7)
N_LOOKUP      = random.randint(6, 10)

ZONES   = ["ZONE_A","ZONE_B","ZONE_C","ZONE_D","ZONE_E"]
TYPES   = ["TEMP","PRESSURE","VIBRATION","HUMIDITY","CURRENT","VOLTAGE","FLOW"]
MONTHS  = ["2024-01","2024-03","2024-06","2024-09","2024-12","2025-03"]

sensor_ids   = [f"SN{random.randint(10000,99999)}" for _ in range(N_SENSORS * 2)]
sensor_ids   = list(dict.fromkeys(sensor_ids))[:N_SENSORS]   # unique

sensor_data  = {}
for sid in sensor_ids:
    sensor_data[sid] = {
        "zone":  random.choice(ZONES),
        "type":  random.choice(TYPES),
        "calib": random.choice(MONTHS),
    }

class SeparateChainingMap:
    """
    Hash map with separate chaining.
    Each bucket is a list of [key, value] pairs.
    """
    def __init__(self, capacity=16, load_threshold=0.75):
        self.capacity       = capacity
        self.load_threshold = load_threshold
        self.size           = 0
        self.buckets        = [[] for _ in range(capacity)]
        self.resize_count   = 0
        self.collision_total = 0

    # ── hash function ─────────────────────────────────────────────
    def _hash(self, key):
        """
        Polynomial rolling hash, then mod capacity.
        For a string key: sum(ord(c) * 31^i) % capacity.
        Using Python's built-in hash() gives non-deterministic results
        across sessions (PYTHONHASHSEED), so we use our own.
        """
        h = 0
        for ch in str(key):
            h = (h * 31 + ord(ch)) & 0xFFFFFFFF
        return h % self.capacity

    # ── load factor ───────────────────────────────────────────────
    @property
    def load_factor(self):
        return self.size / self.capacity

    # ── put ───────────────────────────────────────────────────────
    def put(self, key, value):
        """Insert or update. Resize if load factor exceeds threshold."""
        idx = self._hash(key)
        bucket = self.buckets[idx]
        if len(bucket) > 0:
            self.collision_total += 1
        for pair in bucket:
            if pair[0] == key:
                pair[1] = value   # update existing
                return
        bucket.append([key, value])
        self.size += 1
        if self.load_factor > self.load_threshold:
            self._resize()

    # ── get ───────────────────────────────────────────────────────
    def get(self, key, default=None):
        """Return value for key, or default if not found. O(1) avg."""
        idx = self._hash(key)
        for pair in self.buckets[idx]:
            if pair[0] == key:
                return pair[1]
        return default

    # ── delete ────────────────────────────────────────────────────
    def delete(self, key):
        """Remove key if present. Return True if removed."""
        idx = self._hash(key)
        bucket = self.buckets[idx]
        for i, pair in enumerate(bucket):
            if pair[0] == key:
                bucket.pop(i)
                self.size -= 1
                return True
        return False

    # ── resize ────────────────────────────────────────────────────
    def _resize(self):
        """Double capacity and rehash all entries. O(n)."""
        old_buckets    = self.buckets
        self.capacity  = self.capacity * 2
        self.buckets   = [[] for _ in range(self.capacity)]
        self.size      = 0
        self.resize_count += 1
        for bucket in old_buckets:
            for key, value in bucket:
                self.put(key, value)

    # ── stats ─────────────────────────────────────────────────────
    def bucket_lengths(self):
        return [len(b) for b in self.buckets]

    def max_chain(self):
        return max(self.bucket_lengths())

    def empty_buckets(self):
        return sum(1 for b in self.buckets if len(b) == 0)

reg = SeparateChainingMap(capacity=INIT_CAPACITY, load_threshold=LOAD_THRESH)

print(f"Hash map parameters:")
print(f"  Initial capacity  : {INIT_CAPACITY}")
print(f"  Load threshold    : {LOAD_THRESH}")
print(f"  Sensors to insert : {N_SENSORS}")
print()

# Insert all sensors
for sid in sensor_ids:
    reg.put(sid, sensor_data[sid])

print(f"After inserting {N_SENSORS} sensors:")
print(f"  Final capacity    : {reg.capacity}")
print(f"  Stored entries    : {reg.size}")
print(f"  Load factor       : {reg.load_factor:.3f}")
print(f"  Resize events     : {reg.resize_count}")
print(f"  Total collisions  : {reg.collision_total}")
print(f"  Longest chain     : {reg.max_chain()}")
print(f"  Empty buckets     : {reg.empty_buckets()} / {reg.capacity}")


## Step 2 — Lookup and delete operations

In [ ]:
# Lookup: some keys present, some not
lookup_targets = random.sample(sensor_ids, N_LOOKUP // 2)
missing_ids    = [f"SN{random.randint(10000,99999)}" for _ in range(N_LOOKUP // 2)]
random.shuffle(missing_ids)
all_lookups    = lookup_targets + missing_ids
random.shuffle(all_lookups)

print(f"Lookup results ({len(all_lookups)} queries):")
print(f"  {'Sensor ID':<10}  {'Found':>6}  {'Zone':<8}  {'Type':<12}  Calibrated")
print(f"  {'─'*10}  {'─'*6}  {'─'*8}  {'─'*12}  {'─'*12}")
hits, misses = 0, 0
for sid in all_lookups:
    rec = reg.get(sid)
    if rec:
        hits += 1
        print(f"  {sid:<10}  {'YES':>6}  {rec['zone']:<8}  {rec['type']:<12}  {rec['calib']}")
    else:
        misses += 1
        print(f"  {sid:<10}  {'NO':>6}  {'—':<8}  {'—':<12}  —")

print(f"\n  Hits  : {hits}   Misses: {misses}")

# Delete some sensors
to_delete = random.sample(sensor_ids, N_DELETE)
print(f"\nDeleting {N_DELETE} sensors: {to_delete}")
for sid in to_delete:
    ok = reg.delete(sid)
    print(f"  delete({sid}): {'removed ✅' if ok else 'not found ❌'}")

print(f"\n  Registry size after deletion: {reg.size}")
print(f"  Load factor after deletion : {reg.load_factor:.3f}")


## Step 3 — Bucket distribution analysis

In [ ]:
lengths = reg.bucket_lengths()
from collections import Counter
dist = Counter(lengths)

print(f"Bucket chain-length distribution (capacity={reg.capacity}):")
print(f"  {'Chain len':>10}  {'Buckets':>8}  {'% of capacity':>15}  Bar")
print(f"  {'─'*10}  {'─'*8}  {'─'*15}  {'─'*20}")
for length in sorted(dist):
    count = dist[length]
    pct   = count / reg.capacity * 100
    bar   = "█" * count
    print(f"  {length:10d}  {count:8d}  {pct:14.1f}%  {bar}")

print()
occupied = reg.capacity - reg.empty_buckets()
avg_chain = reg.size / occupied if occupied else 0
print(f"  Occupied buckets  : {occupied} / {reg.capacity}")
print(f"  Avg chain (occupied): {avg_chain:.2f}")
print(f"  Max chain length  : {reg.max_chain()}")
print(f"  Collisions total  : {reg.collision_total}")

# Verify all remaining sensors are retrievable
survivors = [sid for sid in sensor_ids if sid not in to_delete]
all_found = all(reg.get(sid) is not None for sid in survivors)
print(f"\n  All survivors retrievable: {'✅ YES' if all_found else '❌ NO'}")
print(f"  Survivors checked: {len(survivors)}")


## Step 4 — Compare with Python's built-in dict

In [ ]:
import time

# Rebuild both maps with same data
custom = SeparateChainingMap(capacity=INIT_CAPACITY, load_threshold=LOAD_THRESH)
builtin = {}

N_TIMING = 5000
timing_ids  = [f"SN{random.randint(10000,99999)}" for _ in range(N_TIMING)]
timing_vals = [{"v": i} for i in range(N_TIMING)]

# Custom map timing
t0 = time.perf_counter()
for k, v in zip(timing_ids, timing_vals):
    custom.put(k, v)
for k in timing_ids:
    custom.get(k)
t1 = time.perf_counter()
custom_ms = round((t1 - t0) * 1000, 2)

# Built-in dict timing
t0 = time.perf_counter()
for k, v in zip(timing_ids, timing_vals):
    builtin[k] = v
for k in timing_ids:
    builtin.get(k)
t1 = time.perf_counter()
builtin_ms = round((t1 - t0) * 1000, 2)

print(f"Performance comparison ({N_TIMING} insert + {N_TIMING} lookup):")
print(f"  Custom hash map   : {custom_ms:8.2f} ms")
print(f"  Python dict       : {builtin_ms:8.2f} ms")
ratio = round(custom_ms / builtin_ms, 1) if builtin_ms > 0 else 0
print(f"  Ratio (custom/builtin): {ratio}×")
print()
print(f"Custom map final stats:")
print(f"  Capacity          : {custom.capacity}")
print(f"  Load factor       : {custom.load_factor:.3f}")
print(f"  Resize events     : {custom.resize_count}")
print(f"  Max chain length  : {custom.max_chain()}")


## 📋 Final Report

In [ ]:
W = 56
print("╔" + "═"*W + "╗")
print("║" + " SENSOR ID REGISTRY (HASH MAP) — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  HASH MAP PARAMETERS" + " "*(W-21) + "║")
print(f"║  {'Initial capacity':<26}: {INIT_CAPACITY:<26}║")
print(f"║  {'Load threshold':<26}: {LOAD_THRESH:<26}║")
print(f"║  {'Sensors inserted':<26}: {N_SENSORS:<26}║")
print("╠" + "═"*W + "╣")
print("║  AFTER INSERTIONS" + " "*(W-18) + "║")
print(f"║  {'Final capacity':<26}: {reg.capacity:<26}║")
print(f"║  {'Load factor':<26}: {reg.load_factor:.3f}{'':<22}║")
print(f"║  {'Resize events':<26}: {reg.resize_count:<26}║")
print(f"║  {'Total collisions':<26}: {reg.collision_total:<26}║")
print(f"║  {'Max chain length':<26}: {reg.max_chain():<26}║")
print(f"║  {'Empty buckets':<26}: {reg.empty_buckets()} / {reg.capacity}{'':<16}║")
print("╠" + "═"*W + "╣")
print("║  LOOKUP & DELETE" + " "*(W-17) + "║")
print(f"║  {'Lookup hits':<26}: {hits:<26}║")
print(f"║  {'Lookup misses':<26}: {misses:<26}║")
print(f"║  {'Deleted sensors':<26}: {N_DELETE:<26}║")
print(f"║  {'Survivors retrievable':<26}: {'YES ✅' if all_found else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  PERFORMANCE" + " "*(W-13) + "║")
print(f"║  {'Custom map time':<26}: {custom_ms} ms{'':<20}║")
print(f"║  {'Python dict time':<26}: {builtin_ms} ms{'':<20}║")
print(f"║  {'Overhead ratio':<26}: {ratio}×{'':<24}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which hash table concept did you find most important, and why?**
Pick one technique (collision resolution, load factor, LRU cache, rolling hash…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
